In [1]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.inspection import permutation_importance
#import featuretools as ft
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from autofeat import AutoFeatRegressor
import matplotlib.pyplot as plt
from src.preprocessing import scale_features, add_outlier_flags, target_feature_split, one_hot_encoding, create_labels_for_classification
from src.feature_engineering import (create_vocal_instrumental_ratio, create_energy_rhythm_interaction, create_moodscore_bins
                                    ,create_vocal_energy_ratio,create_energy_acoustic_ratio,create_mood_rhythm_interaction,create_live_track_interaction
                                    ,create_classification_model)


In [2]:
# Input Data
df_train = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")
df_sample_submission = pd.read_csv("../data/raw/sample_submission.csv")

In [3]:
flag_autofeat = False


In [4]:
if flag_autofeat:
    X = df_train.drop(columns=["BeatsPerMinute", "id"])
    y = df_train["BeatsPerMinute"]

    X_sample = X.sample(150000, random_state=42)
    y_sample = y.loc[X_sample.index]

    afreg = AutoFeatRegressor(
        verbose=1, 
        featsel_runs=1 
    )

    #X_auto = afreg.fit_transform(X, y)
    X_auto = afreg.fit_transform(X_sample, y_sample)

    print("Original Shape:", X_sample.shape)
    print("Expanded Shape:", X_auto.shape)
    print(X_auto.head())

In [5]:
#preprocessing
df_train, scaler = scale_features(df_train, ['AudioLoudness','TrackDurationMs'])
df_train, _ = add_outlier_flags(df_train, ['RhythmScore','AudioLoudness','VocalContent','AcousticQuality','InstrumentalScore','LivePerformanceLikelihood','TrackDurationMs'])

df_test, _ = scale_features(df_test,  ['AudioLoudness','TrackDurationMs'], scaler)
df_test, _ = add_outlier_flags(df_test, ['RhythmScore','AudioLoudness','VocalContent','AcousticQuality','InstrumentalScore','LivePerformanceLikelihood','TrackDurationMs'])

In [6]:
# feature engineering
df_train = create_vocal_instrumental_ratio(df_train)
df_train = create_energy_rhythm_interaction(df_train)
df_train = create_moodscore_bins(df_train)
df_train = create_vocal_energy_ratio(df_train)
df_train = create_energy_acoustic_ratio(df_train)
df_train = create_mood_rhythm_interaction(df_train)
df_train = create_live_track_interaction(df_train)


df_test = create_vocal_instrumental_ratio(df_test)
df_test = create_energy_rhythm_interaction(df_test)
df_test = create_moodscore_bins(df_test)
df_test = create_vocal_energy_ratio(df_test)
df_test = create_energy_acoustic_ratio(df_test)
df_test = create_mood_rhythm_interaction(df_test)
df_test = create_live_track_interaction(df_test)

In [7]:
#also preprocessing
df_train, encoder = one_hot_encoding(df_train, 'MoodScore_bins')
df_test, _ = one_hot_encoding(df_test, 'MoodScore_bins', encoder)

In [8]:
#Definition of X and y
X_train, y_train = target_feature_split(df=df_train, target="BeatsPerMinute", exclude_cols=['id'])
X_test, y_test = target_feature_split(df=df_test, exclude_cols=['id'])

In [9]:
RANDOM_STATE = 42
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [10]:
train_models = {
    "dummy": DummyRegressor(strategy="mean"),
    "ridge": Ridge(random_state=RANDOM_STATE),
    "rf": RandomForestRegressor(max_depth=12,min_samples_leaf=5, n_jobs=-1,random_state=RANDOM_STATE),
    "lgbm": LGBMRegressor(n_jobs=-1, random_state=RANDOM_STATE)
}

oof = {name: np.zeros(len(X_train)) for name in train_models.keys()}

models = {}

for model in train_models.keys():
    if os.path.exists(f"../models/{model}_v1_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../data/processed/{model}_oof_v1_seed{RANDOM_STATE}.npy"):
        models[f'{model}'] = joblib.load(f"../models/{model}_v1_seed{RANDOM_STATE}.pkl")
        oof[f'{model}'] = np.load(f"../data/processed/{model}_oof_v1_seed{RANDOM_STATE}.npy")
        del train_models[f'{model}']

In [11]:
for i, (train_index, val_index) in enumerate(kf.split(X_train, y_train)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Validation:  index={val_index}")
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

    for model in train_models.keys():
        print(f"{model}")
        train_models[f"{model}"].fit(X_tr, y_tr)
        oof[f"{model}"][val_index] = train_models[f"{model}"].predict(X_val)

for model in train_models.keys():
    joblib.dump(train_models[model], f"../models/{model}_v1_seed{RANDOM_STATE}.pkl")
    np.save(f"../data/processed/{model}_oof_v1_seed{RANDOM_STATE}.npy", oof[f'{model}'])
    models[f"{model}"] = train_models[f"{model}"]

Fold 0:
  Train: index=[     0      1      3 ... 524159 524160 524162]
  Validation:  index=[     2      6      7 ... 524146 524161 524163]
dummy
ridge
rf
lgbm
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.046337 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3841
[LightGBM] [Info] Number of data points in the train set: 419331, number of used features: 23
[LightGBM] [Info] Start training from score 119.056554
Fold 1:
  Train: index=[     1      2      3 ... 524160 524161 524163]
  Validation:  index=[     0     11     16 ... 524153 524159 524162]
dummy
ridge
rf
lgbm
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016271 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3841
[LightGBM] 

In [12]:
#baseline rmse's and mae's
#dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
#ridge RMSE: 26.466341374596727 MAE: 21.19792095039579
#rf RMSE: 26.465788481708348 MAE: 21.197653869516085
#lgbm RMSE: 26.467350811600735 MAE: 21.198498087056862

#first try with new engineered features
#baseline rmse's and mae's
#dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
#ridge RMSE: 26.466341374596727 MAE: 21.19792095039579
#rf RMSE: 26.465788481708348 MAE: 21.197653869516085
#lgbm RMSE: 26.467350811600735 MAE: 21.198498087056862

#first try


In [13]:

rmses = {}
for name, preds in oof.items():
    print(name, "RMSE:", root_mean_squared_error(y_train, preds), "MAE:", mean_absolute_error(y_train, preds))
    rmses

dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
ridge RMSE: 26.465385825000094 MAE: 21.197030571261287
rf RMSE: 26.465305023507657 MAE: 21.196963534692447
lgbm RMSE: 26.467526741672962 MAE: 21.198457247155215


In [14]:
dict_importances = {}
for model in models.keys():
    print(model)
    perm = permutation_importance(models[model], X_train, y_train, scoring="neg_root_mean_squared_error", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    importance_df = pd.DataFrame({"feature": X_train.columns, "perm_importance": perm["importances_mean"]})
    importance_df = importance_df.sort_values("perm_importance", ascending=False)
    dict_importances[model] = importance_df

dummy
ridge
rf
lgbm


In [15]:
#stack = StackingRegressor(
#    estimators=[
#        ("ridge", models['ridge']),
#        ("rf", models['rf']),
#        ("lgbm", models['lgbm'])
#    ],
#    final_estimator=Ridge(),
#    passthrough=True,
#    n_jobs=-1
#)

#scores = cross_val_score(stack, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")
#print("Stacking RMSE:", -np.mean(scores))


In [16]:
if not os.path.exists(f"../results/first_random_forest_submission.csv"):
    #Random Forest submission
    y_test_pred = models['rf'].predict(X_test)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_random_forest_submission.csv", index=False)
    #actually got worse results hehe

In [ ]:
#Creating features with classification
y_train_clf = create_labels_for_classification(y_train)
X_train_v2, clf = create_classification_model(X_train, y_train_clf, kf, RANDOM_STATE)
y_train_v2 = y_train

In [23]:
train_models_v2 = {
    "dummy": DummyRegressor(strategy="mean"),
    "ridge": Ridge(random_state=RANDOM_STATE),
    "rf": RandomForestRegressor(max_depth=12,min_samples_leaf=5, n_jobs=-1,random_state=RANDOM_STATE),
    "lgbm": LGBMRegressor(n_jobs=-1, random_state=RANDOM_STATE)
}

oof = {name: np.zeros(len(X_train_v2)) for name in train_models_v2.keys()}

models_v2 = {}

for model in train_models_v2.keys():
    if os.path.exists(f"../models/{model}_v2_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../data/processed/{model}_oof_v2_seed{RANDOM_STATE}.npy"):
        models_v2[f'{model}'] = joblib.load(f"../models/{model}_v2_seed{RANDOM_STATE}.pkl")
        oof[f'{model}'] = np.load(f"../data/processed/{model}_oof_v2_seed{RANDOM_STATE}.npy")
        del train_models_v2[f'{model}']


for i, (train_index, val_index) in enumerate(kf.split(X_train_v2, y_train_v2)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Validation:  index={val_index}")
    X_tr, X_val = X_train_v2.iloc[train_index], X_train_v2.iloc[val_index]
    y_tr, y_val = y_train_v2.iloc[train_index], y_train_v2.iloc[val_index]

    for model in train_models_v2.keys():
        print(f"{model}")
        train_models_v2[f"{model}"].fit(X_tr, y_tr)
        oof[f"{model}"][val_index] = train_models_v2[f"{model}"].predict(X_val)

for model in train_models_v2.keys():
    joblib.dump(train_models_v2[model], f"../models/{model}_v2_seed{RANDOM_STATE}.pkl")
    np.save(f"../data/processed/{model}_oof_v2_seed{RANDOM_STATE}.npy", oof[f'{model}'])
    models_v2[f"{model}"] = train_models_v2[f"{model}"]

Fold 0:
  Train: index=[     0      1      3 ... 524159 524160 524162]
  Validation:  index=[     2      6      7 ... 524146 524161 524163]
dummy
ridge
rf
lgbm
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.074263 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4606
[LightGBM] [Info] Number of data points in the train set: 419331, number of used features: 26
[LightGBM] [Info] Start training from score 119.056554
Fold 1:
  Train: index=[     1      2      3 ... 524160 524161 524163]
  Validation:  index=[     0     11     16 ... 524153 524159 524162]
dummy
ridge
rf
lgbm
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017933 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4606
[LightGBM] 

In [24]:
dict_importances = {}
for model in models_v2.keys():
    print(model)
    perm = permutation_importance(models_v2[model], X_train_v2, y_train_v2, scoring="neg_root_mean_squared_error", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    importance_df = pd.DataFrame({"feature": X_train_v2.columns, "perm_importance": perm["importances_mean"]})
    importance_df = importance_df.sort_values("perm_importance", ascending=False)
    dict_importances[model] = importance_df

dummy
ridge
rf
lgbm


In [25]:
dict_importances['ridge']

,feature,perm_importance
20,mood_rhythm_interaction,4.902001e-03
22,MoodScore_bins_low,4.075755e-03
21,live_track_interaction,2.163961e-03
23,MoodScore_bins_medium,1.474552e-03
24,proba_slow,1.048913e-03
18,vocal_energy_ratio,6.839092e-04
2,VocalContent,6.778439e-04
26,proba_fast,5.926350e-04
6,MoodScore,5.392569e-04
8,Energy,1.686728e-04


In [26]:
dict_importances['rf']

,feature,perm_importance
24,proba_slow,7.415966e-02
21,live_track_interaction,6.717005e-02
6,MoodScore,6.488419e-02
20,mood_rhythm_interaction,5.109140e-02
7,TrackDurationMs,4.725735e-02
0,RhythmScore,3.889076e-02
1,AudioLoudness,3.528426e-02
26,proba_fast,3.378924e-02
19,energy_acoustic_ratio,3.347493e-02
5,LivePerformanceLikelihood,3.212411e-02


In [27]:
dict_importances['lgbm']

,feature,perm_importance
6,MoodScore,0.047583
2,VocalContent,0.045234
20,mood_rhythm_interaction,0.042097
18,vocal_energy_ratio,0.034778
24,proba_slow,0.033145
21,live_track_interaction,0.032940
8,Energy,0.030247
0,RhythmScore,0.029947
17,energy_rhythm_interaction,0.029509
7,TrackDurationMs,0.027525


In [ ]:
rmses = {}
for name, preds in oof.items():
    print(name, "RMSE:", root_mean_squared_error(y_train_v2, preds), "MAE:", mean_absolute_error(y_train_v2, preds))
    rmses

dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
ridge RMSE: 26.464581033700515 MAE: 21.196263361285556
rf RMSE: 26.46161412666076 MAE: 21.19429907984696
lgbm RMSE: 26.467051728056493 MAE: 21.197983771897544


In [ ]:
if not os.path.exists(f"../results/second_random_forest_submission.csv"):
    #Random Forest submission
    X_test_v2 = create_classification_model(X_test, clf)
    y_test_pred = models_v2['rf'].predict(X_test_v2)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/second_random_forest_submission.csv", index=False)
    #actually got worse results hehe

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- proba_fast
- proba_medium
- proba_slow


In [31]:
dict_error_by_bin = {}
for model in models_v2.keys():
    df_resid = pd.DataFrame({"y_true": y_train_v2, "y_pred": oof[model]})
    
    df_resid["BPM_bin"] = pd.qcut(df_resid["y_true"], q=5, labels=[f"Q1", "Q2", "Q3", "Q4", "Q5"])
    
    error_by_bin = df_resid.groupby("BPM_bin").apply(lambda d: root_mean_squared_error(d.y_true, d.y_pred))
    dict_error_by_bin[model] = error_by_bin

C:\Users\menez\AppData\Local\Temp\ipykernel_15016\2652247669.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  error_by_bin = df_resid.groupby("BPM_bin").apply(lambda d: root_mean_squared_error(d.y_true, d.y_pred))
C:\Users\menez\AppData\Local\Temp\ipykernel_15016\2652247669.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  error_by_bin = df_resid.groupby("BPM_bin").apply(lambda d: root_mean_squared_error(d.y_true, d.y_pred))
C:\Users\menez\AppData\Local\Temp\ipykernel_15016\2652247669.py:7: FutureWarning: Th

In [32]:
dict_error_by_bin

{'dummy': BPM_bin
 Q1    38.537388
 Q2    15.007049
 Q3     3.951164
 Q4    14.614545
 Q5    39.538506
 dtype: float64,
 'ridge': BPM_bin
 Q1    38.522524
 Q2    15.013749
 Q3     3.981901
 Q4    14.616621
 Q5    39.534894
 dtype: float64,
 'rf': BPM_bin
 Q1    38.498686
 Q2    15.024274
 Q3     4.060065
 Q4    14.629971
 Q5    39.531302
 dtype: float64,
 'lgbm': BPM_bin
 Q1    38.509849
 Q2    15.037905
 Q3     4.084525
 Q4    14.631043
 Q5    39.530529
 dtype: float64}

In [ ]:
##Stacking submission
#y_test_pred = stack.predict(X_test)
#df_sample_submission['BeatsPerMinute'] = y_test_pred
#df_sample_submission.to_csv("../results/first_stack_submission.csv", index=False)